# Archived experiment source: EndekGAN_EVALUATION_NOTEBOOK.ipynb
All historical cell outputs were cleared for repository publication. The original SHA-256 is recorded in docs/SOURCE_MANIFEST.json.
WARNING: direct Run All may train models, download dependencies, or create files. Use the master orchestrator first. The public folder excludes copyrighted source images and model weights.


# EndekGAN-T2I — Additional image evaluation

**Goal.** Evaluate existing outputs of Category GAN, Hash-Text, Transformer, Seam-Aware EndekGAN-T2I, ablations, and adapted DF-GAN–MiniLM.

This notebook **does not train models or modify original checkpoints**. It uses the same held-out test split and matches display images by source filename and exact prompt. The DF-GAN architecture uses independent latent vectors.

In [ ]:
from pathlib import Path
import os, sys, subprocess, pandas as pd
from IPython.display import display, Image as IPyImage
ROOT = Path(os.environ.get("ENDEKGAN_PROJECT_ROOT", "../EndekGAN-T2I")).resolve()
PROJECT = ROOT / "EVALUATION_EXTENSION_20261105"
SCRIPT = PROJECT / "endek_evaluate_existing_outputs.py"
assert SCRIPT.is_file(), SCRIPT
print("Evaluation code:", SCRIPT)

## Step 1 — Quick validation (seed 42)

Creates model-comparison panels, repeated-tile panels, image-level diagnostics, and **blank** blind expert-rating forms. Results are kept in `RESULTS/SEED_42`. The program uses 16 images per class, per model by default.

In [ ]:
RUN_QUICK = False
if RUN_QUICK:
    subprocess.run([sys.executable, str(SCRIPT), "--seeds", "42", "--metric-per-class", "16"], check=True)
else:
    print("Skipped; existing results remain unchanged")

## Step 2 — Inspect the actual generated-image comparison

The first panel shows real Endek test images beside all model variants. The second shows 2×2 repeated tiles for standard Transformer, Proposed Seam-Aware, and DF-GAN–MiniLM.

In [ ]:
for name in ["FIG_MODEL_COMPARISON.png", "FIG_SEAM_TILE_COMPARISON.png"]:
    path = PROJECT / "RESULTS" / "SEED_42" / name
    if path.exists():
        print(name)
        display(IPyImage(filename=str(path), width=1200))

## Step 3 — Inspect per-image and class-wise diagnostics

**Seam gap:** smaller means closer opposite-edge pixel values, not necessarily clearer ornaments. **Detail energy / periodicity proxy:** exploratory indicators, not validated motif or weaving metrics.

In [ ]:
p = PROJECT / "RESULTS" / "SEED_42" / "EVAL_CLASS_SUMMARY.csv"
if p.exists():
    display(pd.read_csv(p).round(5))

## Step 4 — Full three-seed evaluation (optional)

For paper-quality descriptive data, run all three seeds and all 128 generated images per class, per model. This is more expensive than the quick test but it uses already-saved PNGs and **does not retrain**.

In [ ]:
RUN_ALL_THREE_SEEDS = False  # Set True intentionally when ready
if RUN_ALL_THREE_SEEDS:
    subprocess.run([sys.executable, str(SCRIPT), "--seeds", "42", "123", "2025", "--metric-per-class", "128"], check=True)
else:
    print("Full three-seed evaluation not run in this notebook session")

## Step 5 — Multilingual CLIP diagnostic (optional)

Requires pretrained image encoder `sentence-transformers/clip-ViT-B-32` and multilingual text encoder `sentence-transformers/clip-ViT-B-32-multilingual-v1`. Model downloads may be necessary. This is **not a validated prompt-image alignment accuracy**: source captions were generated from templates and have not been annotated image-by-image by Endek experts.

In [ ]:
RUN_SEMANTIC_DIAGNOSTIC = False  # This can load/download large models
if RUN_SEMANTIC_DIAGNOSTIC:
    subprocess.run([sys.executable, str(SCRIPT), "--seeds", "42", "--metric-per-class", "16", "--semantic"], check=True)
else:
    print("Semantic diagnostic not run; no CLIP scores claimed")

## Step 6 — Blinded expert assessment

Give experts the image files under `BLINDED_IMAGES` and `EXPERT_BLIND_RATING_BLANK.csv`. Do **not** share `EXPERT_PRIVATE_MAPPING_DO_NOT_SHARE.csv`. Collect independent ratings, then analyze only rows actually scored. Never prepopulate or invent expert ratings.